# 🟠 PR Score — Automação Loft

**Como usar este notebook:**
1. **Célula 1** — Instala dependências (rode uma vez)
2. **Célula 2** — Configure sua chave Gemini e o ID do Dashboard
3. **Célula 3** — Autentica Google Sheets e carrega lista Tier 1
4. **Célula 4** — Faça upload da planilha da Clipadora e limpe os dados
5. **Célula 5** — IA avalia protagonismo de cada matéria
6. **Célula 6** — Escreve resultados no Dashboard
7. **Célula 7** — Gera texto do WhatsApp e HTML do e-mail

---
⚠️ **Antes de rodar:** certifique-se de ter a chave da API Gemini e o ID da planilha Dashboard.

In [ ]:
# ============================================================
# CÉLULA 1 — Instalar dependências
# ============================================================
!pip install gspread google-auth google-generativeai openpyxl requests beautifulsoup4 --quiet

import pandas as pd
import gspread
import google.generativeai as genai
import requests
import json
import time
import re
from datetime import datetime, date
from bs4 import BeautifulSoup
from google.colab import auth, files
from google.auth import default
from io import BytesIO

print('✅ Dependências instaladas e importadas com sucesso!')

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração
# Preencha os campos abaixo antes de continuar
# ============================================================

# Chave da API Gemini (obter em https://aistudio.google.com)
GEMINI_API_KEY = ""  # ← Cole sua chave aqui

# ID do Google Sheets Dashboard
# (parte da URL entre /d/ e /edit)
DASHBOARD_SHEET_ID = ""  # ← Cole o ID aqui

# Data do clipping (padrão: hoje)
DATA_CLIPPING = date.today().strftime("%d/%m/%Y")
# Para outra data, descomente e edite:
# DATA_CLIPPING = "20/06/2026"

# Modelo Gemini
GEMINI_MODEL = "gemini-2.0-flash"

# Validação
assert GEMINI_API_KEY, "❌ Configure a GEMINI_API_KEY!"
assert DASHBOARD_SHEET_ID, "❌ Configure o DASHBOARD_SHEET_ID!"

print(f"✅ Configuração OK — clipping de {DATA_CLIPPING}")

In [ ]:
# ============================================================
# CÉLULA 3 — Autenticação e carregamento da lista Tier 1
# ============================================================

# --- Autenticar Google Sheets ---
print("🔐 Autenticando com Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Google Sheets autenticado!")

# --- Inicializar Gemini ---
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)
# Teste rápido
test = model.generate_content("Responda apenas: OK")
print(f"✅ Gemini conectado! Resposta teste: {test.text.strip()}")

# --- Carregar lista Tier 1 ---
print("\n📋 Faça upload da planilha Tier 1 (Hackaton_Tier_1__Loft_1.xlsx):")
uploaded_tier1 = files.upload()
tier1_filename = list(uploaded_tier1.keys())[0]

df_tier1_raw = pd.read_excel(BytesIO(uploaded_tier1[tier1_filename]), header=None)

# Planilha Tier 1 tem veículos espalhados em múltiplas colunas — achatar tudo
tier1_lista = set()
for col in df_tier1_raw.columns:
    for val in df_tier1_raw[col].dropna():
        v = str(val).strip()
        if v:
            tier1_lista.add(v.lower())

print(f"✅ Lista Tier 1 carregada: {len(tier1_lista)} veículos")

# Veículos com paywall (exigem olhar humano)
PAYWALL_VEICULOS = ["valor econômico", "jornal do comércio"]

# Canais a remover da Clipadora
CANAIS_EXCLUIR = ["coelho da fonseca", "lopes", "assuntos de interesse"]

# Equivalências de marca
EQUIVALENCIAS = {
    "olx brasil": "ZAP",
    "olx": "ZAP",
    "viva real": "ZAP",
    "imovelweb": "QuintoAndar",
    "foxter": "Loft"
}

# Temas disponíveis (base histórica)
TEMAS_BASE = [
    "Mercado Imobiliário", "Precificação", "Locação", "Compra e Venda",
    "Tecnologia e Produto", "Porta-voz", "Pesquisas e Tendências",
    "Expansão e Negócios", "Crise e Reputação", "Mercado de Capitais",
    "Urbanismo e Cidades", "Financiamento e Crédito", "ESG", "RH e Cultura"
]

print("\n✅ Configuração completa — pode ir para a Célula 4!")

In [ ]:
# ============================================================
# CÉLULA 4 — Upload e limpeza da planilha da Clipadora
# ============================================================

print("📂 Faça upload da planilha da Clipadora (.xlsx):")
uploaded_clip = files.upload()
clip_filename = list(uploaded_clip.keys())[0]

# Cabeçalhos reais estão na linha 4 (índice 3)
df_clip = pd.read_excel(
    BytesIO(uploaded_clip[clip_filename]),
    sheet_name="Matérias",
    header=3  # linha 4
)

total_original = len(df_clip)
print(f"\n📊 Total de matérias no clipping: {total_original}")

# Renomear colunas chave para nomes seguros
df_clip = df_clip.rename(columns={
    "Matéria": "titulo",
    "Veículo": "veiculo",
    "Data de Indexação": "data_indexacao",
    "Canal": "canal",
    "URL da Fonte": "url_fonte",
    "Link": "link_clipadora",
    "Release do Cliente": "release_cliente",
    "Matéria Repetida": "materia_repetida",
    "Mídia": "midia"
})

# --- Passo 1: Remover canais excluídos ---
mask_canal = df_clip["canal"].str.strip().str.lower().isin(CANAIS_EXCLUIR)
n_canal = mask_canal.sum()
df_clip = df_clip[~mask_canal].copy()
print(f"🗑️  Removidas {n_canal} linhas de canais excluídos")

# --- Passo 2: Filtro Tier 1 ---
def classificar_tier1(veiculo):
    """Retorna 'tier1', 'revisar' ou 'excluir'"""
    if pd.isna(veiculo):
        return "excluir"
    v = str(veiculo).strip().lower()
    if v in tier1_lista:
        return "tier1"
    # Verificação fuzzy: se o nome do veículo contém ou está contido em algum Tier 1
    for t1 in tier1_lista:
        if (v in t1 or t1 in v) and len(v) > 4:
            return "revisar"
    return "excluir"

df_clip["tier1_status"] = df_clip["veiculo"].apply(classificar_tier1)

df_tier1 = df_clip[df_clip["tier1_status"] == "tier1"].copy()
df_revisar_tier1 = df_clip[df_clip["tier1_status"] == "revisar"].copy()
n_excluidos_tier = (df_clip["tier1_status"] == "excluir").sum()

print(f"✅ Tier 1 confirmados: {len(df_tier1)}")
print(f"⚠️  Para revisão de Tier 1 (match parcial): {len(df_revisar_tier1)}")
print(f"🗑️  Não-Tier 1 removidos: {n_excluidos_tier}")

# Adicionar veículos para revisão com flag
df_revisar_tier1["obs_limpeza"] = "⚠️ Verificar se é Tier 1 — match parcial no nome do veículo"

# --- Passo 3: Remover press releases pagos ---
mask_pago = (
    df_tier1["release_cliente"].notna() & 
    (df_tier1["release_cliente"].astype(str).str.strip() != "")
)
n_pago = mask_pago.sum()
df_tier1 = df_tier1[~mask_pago].copy()
print(f"🗑️  Removidos {n_pago} press releases pagos")

# --- Passo 4: Detectar títulos repetidos em múltiplos veículos no mesmo dia ---
df_tier1["data_str"] = pd.to_datetime(df_tier1["data_indexacao"], errors="coerce").dt.strftime("%d/%m/%Y")
contagem_titulo = df_tier1.groupby(["titulo", "data_str"])["veiculo"].transform("count")
mask_duplicado = (df_tier1["materia_repetida"].notna() & 
                  (df_tier1["materia_repetida"].astype(str).str.strip() != "")) | \
                 (contagem_titulo >= 3)
n_dup = mask_duplicado.sum()
df_tier1 = df_tier1[~mask_duplicado].copy()
print(f"🗑️  Removidos {n_dup} duplicatas/press releases em massa")

# --- Passo 5: Corrigir data ---
df_tier1["data_formatada"] = pd.to_datetime(
    df_tier1["data_indexacao"], errors="coerce"
).dt.strftime("%d/%m/%Y")
df_tier1["data_formatada"] = df_tier1["data_formatada"].fillna(DATA_CLIPPING)

# --- Passo 6: Empresa via Canal e equivalências ---
def identificar_empresa(row):
    canal = str(row.get("canal", "")).lower()
    veiculo = str(row.get("veiculo", "")).lower().strip()
    # Verificar equivalências de marca primeiro
    for marca, empresa in EQUIVALENCIAS.items():
        if marca in veiculo:
            return empresa
    # Identificar empresa pelo canal
    if "loft" in canal or "foxter" in canal:
        return "Loft"
    if "quintoandar" in canal or "imovelweb" in canal:
        return "QuintoAndar"
    if "zap" in canal or "olx" in canal or "viva real" in canal:
        return "ZAP"
    if "superlógica" in canal or "superlogica" in canal:
        return "Superlógica"
    if "creditas" in canal:
        return "Creditas"
    if "porto seguro" in canal:
        return "Porto Seguro"
    return str(row.get("canal", "Desconhecido"))

df_tier1["empresa"] = df_tier1.apply(identificar_empresa, axis=1)

# --- Passo 7: Separar off-topic para Double Check ---
# Heurística: OLX/ZAP com segmentos não-imobiliários, Porto Seguro em veículos, etc.
keywords_off_topic = ["automóvel", "carro", "veículo", "celular", "smartphone", 
                       "seguros de vida", "previdência"]

def provavelmente_off_topic(titulo):
    t = str(titulo).lower()
    return any(k in t for k in keywords_off_topic)

mask_off = df_tier1["titulo"].apply(provavelmente_off_topic)
df_double_check = df_tier1[mask_off].copy()
df_main = df_tier1[~mask_off].copy().reset_index(drop=True)

print(f"\n📋 RESUMO DA LIMPEZA")
print(f"   Entrada:          {total_original} matérias")
print(f"   Após limpeza:     {len(df_main)} matérias para avaliação")
print(f"   Double Check:     {len(df_double_check)} matérias off-topic para revisão humana")
print(f"   Tier 1 incerto:   {len(df_revisar_tier1)} veículos para validação")

if len(df_revisar_tier1) > 0:
    print(f"\n⚠️  Veículos para validação de Tier 1:")
    for v in df_revisar_tier1["veiculo"].unique():
        print(f"   - {v}")

In [ ]:
# ============================================================
# CÉLULA 5 — Avaliação de protagonismo com Gemini
# ============================================================

PROMPT_TEMPLATE = """Você é um analista sênior de PR da Loft. Avalie a matéria abaixo com base nos critérios do PR Score.

=== MATÉRIA ===
Título: {titulo}
Veículo: {veiculo}
Empresa monitorada: {empresa}
Texto disponível: {texto}

=== LÓGICA DE AVALIAÇÃO — SIGA ESTA ORDEM ===

PASSO 1 — Critérios gatilho (C9 a C19):
Verifique se ALGUM dos critérios abaixo é verdadeiro para a empresa monitorada ({empresa}):

C9 - Declarações de porta-vozes: há ao menos 3 frases atribuídas a porta-voz da empresa?
C10 - Extensão da referência: há ao menos 5 frases ou 8 linhas referenciando a empresa/porta-voz?
C11 - Espaço da matéria: a empresa ocupa ao menos 30% do espaço total da matéria?
C13 - Fonte de dados: a empresa é citada como fonte de dados ou pesquisa? (se usou dado, pontua independente do tamanho da menção)
C16 - Alternância pos/neg: há sentenças positivas e negativas alternadas sobre fatos distintos da empresa?
C17 - Publieditorial disfarçado: parece conteúdo pago/editorial sem identificação clara de publicidade?
C18 - Anúncio governamental: há anúncio do governo (federal, estadual ou municipal de capital) citando a empresa nominalmente?
C19 - Equivalências de marca: a matéria menciona OLX/Viva Real (conta para ZAP), ImovelWeb (conta para QuintoAndar), ou Foxter (conta para Loft)?

Se NENHUM critério C9-C19 for verdadeiro:
  - Verifique C1 (título) e C2 (foto) — se verdadeiros, é Destaque mesmo assim.
  - Caso contrário: protagonismo = Menção. Retorne o JSON e PARE.

Se ALGUM critério C9-C19 for verdadeiro: protagonismo = Destaque. Vá para Passo 2.

PASSO 2 — Pontos extras (só se for Destaque):
Cada critério verdadeiro abaixo = +1 ponto extra:

C1 - Citação no título ou linha-fina: o nome da empresa aparece no título ou chapéu/linha-fina?
C6 - Gráfico/infográfico/box: há elemento visual mencionando a empresa? (sinalize para verificação visual)
C7 - Artigo de executivo: é artigo escrito por executivo da empresa?
C20 - 6+ menções no texto: a empresa é mencionada 6 ou mais vezes?
C21 - CTA para outra matéria: há chamado explícito ('leia mais', 'veja também') linkando outra matéria que cite a empresa?

Critérios que SEMPRE exigem olhar humano (sinalize em obs):
C3, C4 - TV/Rádio: duração e aspas de porta-voz
C5 - Chamadas em redes sociais e home
C8 - Gatilho de análise especial (10+ trechos de porta-voz)
C12 - Lives (mais de 1.000 visualizações)
C14 - Citações em colunas jornalísticas
C15 - Veículo nichado estratégico

Verificação de tom:
- Se for Destaque mas o conteúdo for prejudicial à reputação → Destaque Negativo
- Se for Menção mas prejudicial → Menção Negativa

Dado Proativo vs. Dado de Carona (avalie APENAS se for Destaque E houver dados da empresa):
- Dado Proativo (dado_proativo=true): foco central da matéria gira em torno do dado/estudo fornecido pela empresa
- Dado de Carona (dado_carona=true): o dado surge apenas como apoio em matéria sobre outro assunto

PR Tec+Produto (true apenas para Loft): matéria sobre tecnologia ou produto da Loft?
PR Puro Imobis (true apenas para Loft): matéria sobre mercado imobiliário com Loft como fonte principal?

Temas disponíveis (prefira um deles): {temas_base}
Se precisar criar um novo, sinalize em obs com: NOVO TEMA SUGERIDO: [nome]

RETORNE APENAS O JSON ABAIXO, SEM NENHUM TEXTO ADICIONAL:
{{
  "protagonismo": "<Destaque|Menção|Destaque Negativo|Menção Negativa>",
  "pontuacao_total": <int>,
  "criterios_verdadeiros": [<lista de ints>],
  "tipo": "<Online|Impresso|Rádio|TV|Chamada Online>",
  "pr_tec_produto": <true|false>,
  "pr_puro_imobis": <true|false>,
  "dado_proativo": <true|false>,
  "dado_carona": <true|false>,
  "tema": "<tema macro>",
  "subtema": "<subtema específico>",
  "cidade": "<cidade protagonista ou cidade-sede do veículo>",
  "estado": "<estado protagonista ou estado-sede do veículo>",
  "obs": "<✅ Avaliação completa | ⚠️ Verificar [motivo] | 🔴 Olhar humano obrigatório [motivo] | 🔴 PAYWALL>"
}}"""


def raspar_texto(url, timeout=10):
    """Tenta raspar o texto de uma matéria pelo URL."""
    if pd.isna(url) or not str(url).startswith("http"):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; PRScoreBot/1.0)"}
        resp = requests.get(str(url), headers=headers, timeout=timeout)
        soup = BeautifulSoup(resp.text, "html.parser")
        # Remover scripts e estilos
        for tag in soup(["script", "style", "nav", "header", "footer"]):
            tag.decompose()
        paragrafos = soup.find_all("p")
        texto = " ".join(p.get_text().strip() for p in paragrafos)
        return texto[:4000]  # limitar tokens
    except Exception:
        return ""


def avaliar_materia(row):
    """Avalia uma matéria com Gemini e retorna dict com resultado."""
    titulo = str(row.get("titulo", ""))
    veiculo = str(row.get("veiculo", ""))
    empresa = str(row.get("empresa", ""))

    # Verificar paywall
    for pw in PAYWALL_VEICULOS:
        if pw in veiculo.lower():
            return {
                "protagonismo": "Menção", "pontuacao_total": 0,
                "criterios_verdadeiros": [], "tipo": "Impresso",
                "pr_tec_produto": False, "pr_puro_imobis": False,
                "dado_proativo": False, "dado_carona": False,
                "tema": "", "subtema": "", "cidade": "", "estado": "",
                "obs": f"🔴 PAYWALL — {veiculo}: acessar com login premium e avaliar manualmente"
            }

    # Raspar texto
    texto = raspar_texto(row.get("url_fonte", ""))
    obs_raspagem = ""
    if not texto:
        texto = f"[Texto não disponível — avalie pelo título: {titulo}]"
        obs_raspagem = "⚠️ Texto não raspado — avaliação baseada apenas no título"

    prompt = PROMPT_TEMPLATE.format(
        titulo=titulo,
        veiculo=veiculo,
        empresa=empresa,
        texto=texto,
        temas_base=", ".join(TEMAS_BASE)
    )

    # Chamar Gemini com retry
    for tentativa in range(3):
        try:
            resposta = model.generate_content(prompt)
            texto_resp = resposta.text.strip()
            # Limpar markdown se vier com ```json
            texto_resp = re.sub(r"```json\s*", "", texto_resp)
            texto_resp = re.sub(r"```\s*", "", texto_resp)
            resultado = json.loads(texto_resp)
            if obs_raspagem:
                resultado["obs"] = obs_raspagem + " | " + resultado.get("obs", "")
            return resultado
        except json.JSONDecodeError:
            time.sleep(2)
        except Exception as e:
            if tentativa < 2:
                time.sleep(2 ** (tentativa + 1))
            else:
                return {
                    "protagonismo": "Menção", "pontuacao_total": 0,
                    "criterios_verdadeiros": [], "tipo": "Online",
                    "pr_tec_produto": False, "pr_puro_imobis": False,
                    "dado_proativo": False, "dado_carona": False,
                    "tema": "", "subtema": "", "cidade": "", "estado": "",
                    "obs": f"🔴 Erro na avaliação — revisar manualmente ({str(e)[:80]})"
                }


# --- Rodar avaliação ---
print(f"🤖 Iniciando avaliação de {len(df_main)} matérias com Gemini...")
print(f"   Tempo estimado: ~{len(df_main) * 4 // 60}–{len(df_main) * 6 // 60} minutos\n")

resultados = []
for i, (idx, row) in enumerate(df_main.iterrows()):
    print(f"   [{i+1}/{len(df_main)}] {str(row.get('titulo', ''))[:70]}...")
    resultado = avaliar_materia(row)
    resultados.append(resultado)
    time.sleep(1)  # respeitar limite de 15 RPM do Gemini

# Adicionar resultados ao DataFrame
df_resultados = pd.DataFrame(resultados)
df_main = pd.concat([df_main.reset_index(drop=True), df_resultados], axis=1)

# Resumo
destaques = (df_main["protagonismo"] == "Destaque").sum()
mencoes = (df_main["protagonismo"] == "Menção").sum()
negativos = df_main["protagonismo"].str.contains("Negativo").sum()
humanos = df_main["obs"].str.contains("🔴").sum()

print(f"\n✅ AVALIAÇÃO CONCLUÍDA")
print(f"   Destaques:             {destaques}")
print(f"   Menções:               {mencoes}")
print(f"   Negativos:             {negativos}")
print(f"   Revisão humana (🔴):   {humanos}")

In [ ]:
# ============================================================
# CÉLULA 6 — Escrever resultados no Google Sheets Dashboard
# ============================================================

print("📊 Conectando ao Dashboard...")
sh = gc.open_by_key(DASHBOARD_SHEET_ID)

# --- Aba Matérias ---
ws_materias = sh.worksheet("Matérias")
todos_valores = ws_materias.get_all_values()
ultimo_indice = 0
ultima_linha = len(todos_valores)

# Encontrar último índice para continuar numeração
for row in reversed(todos_valores):
    try:
        ultimo_indice = int(row[0])
        break
    except (ValueError, IndexError):
        continue

print(f"   Último índice no Dashboard: {ultimo_indice}")
print(f"   Escrevendo a partir da linha: {ultima_linha + 1}")

def formatar_linha_dashboard(row, indice):
    """Converte uma linha do DataFrame para o formato do Dashboard (colunas A-W)."""
    titulo = str(row.get("titulo", ""))
    veiculo = str(row.get("veiculo", ""))
    data = str(row.get("data_formatada", DATA_CLIPPING))
    link = str(row.get("link_clipadora", ""))
    mes = data.split("/")[1].lstrip("0") if "/" in data else ""
    modelo_clipping = f"{titulo} — {veiculo}\n{link}"

    return [
        indice,                                          # A - Índice
        mes,                                             # B - Mês
        titulo,                                          # C - Título
        veiculo,                                         # D - Veículo
        data,                                            # E - Data
        str(row.get("canal", "")),                       # F - Canal
        link,                                            # G - Link
        str(row.get("empresa", "")),                     # H - Empresa
        str(row.get("protagonismo", "Menção")),          # I - Protagonismo
        str(row.get("tipo", "Online")),                  # J - Tipo
        str(row.get("pr_tec_produto", False)),           # K - PR Tec+Produto
        str(row.get("pr_puro_imobis", False)),           # L - PR Puro Imobis
        str(row.get("dado_proativo", False)),            # M - Data (proativo)
        str(row.get("dado_carona", False)),              # N - Data Carona
        "",                                              # O - Retranca (humano)
        modelo_clipping,                                 # P - Modelo Clipping
        str(row.get("tema", "")),                        # Q - Tema
        str(row.get("subtema", "")),                     # R - Subtema
        "",                                              # S - Produto
        str(row.get("obs", "")),                         # T - OBS
        "",                                              # U - Chave Subscore (auto)
        str(row.get("cidade", "")),                      # V - Cidade
        str(row.get("estado", ""))                       # W - Estado
    ]

# Montar todas as linhas
novas_linhas = []
for i, (idx, row) in enumerate(df_main.iterrows()):
    linha = formatar_linha_dashboard(row, ultimo_indice + i + 1)
    novas_linhas.append(linha)

# Escrever em batch (mais rápido)
if novas_linhas:
    range_inicio = f"A{ultima_linha + 1}"
    ws_materias.update(range_inicio, novas_linhas, value_input_option="USER_ENTERED")
    print(f"✅ {len(novas_linhas)} matérias escritas na aba Matérias")

# --- Aba Double Check ---
if len(df_double_check) > 0:
    try:
        ws_dc = sh.worksheet("Double Check")
    except gspread.WorksheetNotFound:
        ws_dc = sh.add_worksheet(title="Double Check", rows=500, cols=10)
        ws_dc.update("A1", [["Título", "Veículo", "Data", "Canal", "Empresa", "Link", "OBS"]])

    linhas_dc = []
    for _, row in df_double_check.iterrows():
        linhas_dc.append([
            str(row.get("titulo", "")),
            str(row.get("veiculo", "")),
            str(row.get("data_formatada", DATA_CLIPPING)),
            str(row.get("canal", "")),
            str(row.get("empresa", "")),
            str(row.get("link_clipadora", "")),
            str(row.get("obs_limpeza", "⚠️ Verificar contexto — possível off-topic"))
        ])

    proxima_linha_dc = len(ws_dc.get_all_values()) + 1
    ws_dc.update(f"A{proxima_linha_dc}", linhas_dc, value_input_option="USER_ENTERED")
    print(f"✅ {len(linhas_dc)} matérias escritas na aba Double Check")

print(f"\n🔗 Dashboard: https://docs.google.com/spreadsheets/d/{DASHBOARD_SHEET_ID}/edit")

In [ ]:
# ============================================================
# CÉLULA 7 — Gerar texto do WhatsApp e HTML do e-mail
# ============================================================

EMPRESAS_PRINCIPAIS = ["Loft", "QuintoAndar", "ZAP", "Superlógica"]
CORES_EMPRESA = {
    "Loft": "#FF6B35",
    "QuintoAndar": "#1A73E8",
    "ZAP": "#34A853",
    "Superlógica": "#212121"
}
EMOJIS_EMPRESA = {
    "Loft": "🟠",
    "QuintoAndar": "🔵",
    "ZAP": "🟢",
    "Superlógica": "⚫"
}

# --- Calcular pontuações ---
df_destaques = df_main[df_main["protagonismo"] == "Destaque"]
pontuacoes = {}
for empresa in EMPRESAS_PRINCIPAIS:
    pts = df_destaques[df_destaques["empresa"] == empresa]["pontuacao_total"].sum()
    pontuacoes[empresa] = int(pts) if pts else 0

# Melhor matéria do dia (maior pontuação)
melhor = df_destaques.sort_values("pontuacao_total", ascending=False).head(1)
destaque_texto = ""
if len(melhor) > 0:
    m = melhor.iloc[0]
    destaque_texto = f"{m.get('empresa','')} — {m.get('titulo', '')[:80]} ({m.get('veiculo', '')})"

revisoes_pendentes = df_main["obs"].str.contains("🔴", na=False).sum()

# ============================================================
# OUTPUT 1 — MENSAGEM WHATSAPP
# ============================================================
whatsapp = f"""📊 PR Score — {DATA_CLIPPING}
"""
for empresa in EMPRESAS_PRINCIPAIS:
    emoji = EMOJIS_EMPRESA.get(empresa, "▪️")
    pts = pontuacoes.get(empresa, 0)
    whatsapp += f"{emoji} {empresa}: {pts} pt{'s' if pts != 1 else ''}\n"

if destaque_texto:
    whatsapp += f"\n⚡ Destaque: {destaque_texto}"

if revisoes_pendentes > 0:
    whatsapp += f"\n\n⚠️ {revisoes_pendentes} matéria{'s' if revisoes_pendentes != 1 else ''} aguardam revisão humana"

print("=" * 60)
print("📱 MENSAGEM WHATSAPP — copie e cole no grupo")
print("=" * 60)
print(whatsapp)

# ============================================================
# OUTPUT 2 — HTML DO E-MAIL
# ============================================================
def gerar_secao_empresa(empresa, df_dia):
    cor = CORES_EMPRESA.get(empresa, "#333")
    df_emp = df_dia[
        (df_dia["empresa"] == empresa) & 
        (df_dia["protagonismo"].isin(["Destaque", "Destaque Negativo"]))
    ].sort_values("pontuacao_total", ascending=False)

    if len(df_emp) == 0:
        return f"""
    <div style="margin-bottom:32px">
      <h2 style="color:{cor};border-bottom:2px solid {cor};padding-bottom:6px">{empresa}</h2>
      <p style="color:#888;font-style:italic">Sem destaques hoje.</p>
    </div>"""

    itens = ""
    for _, row in df_emp.iterrows():
        titulo = str(row.get("titulo", ""))
        veiculo = str(row.get("veiculo", ""))
        data = str(row.get("data_formatada", DATA_CLIPPING))
        link = str(row.get("link_clipadora", "#"))
        pts = int(row.get("pontuacao_total", 1))
        prot = str(row.get("protagonismo", ""))
        badge_cor = "#e53935" if "Negativo" in prot else cor
        badge_txt = "Destaque Negativo" if "Negativo" in prot else f"{pts} pt{'s' if pts != 1 else ''}"

        itens += f"""
      <div style="border-left:3px solid {badge_cor};padding:10px 16px;margin-bottom:12px;background:#fafafa">
        <a href="{link}" style="color:#111;font-weight:600;font-size:15px;text-decoration:none">{titulo}</a>
        <div style="margin-top:4px;font-size:13px;color:#666">
          {veiculo} &nbsp;·&nbsp; {data} &nbsp;
          <span style="background:{badge_cor};color:#fff;padding:2px 8px;border-radius:12px;font-size:12px">{badge_txt}</span>
        </div>
      </div>"""

    pts_total = pontuacoes.get(empresa, 0)
    return f"""
    <div style="margin-bottom:32px">
      <h2 style="color:{cor};border-bottom:2px solid {cor};padding-bottom:6px">
        {empresa} <span style="font-size:14px;font-weight:normal;color:#888">{pts_total} ponto{'s' if pts_total != 1 else ''} hoje</span>
      </h2>
      {itens}
    </div>"""

secoes_html = "".join(gerar_secao_empresa(e, df_main) for e in EMPRESAS_PRINCIPAIS)

html_email = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>PR Score — {DATA_CLIPPING}</title>
</head>
<body style="font-family:Arial,sans-serif;max-width:680px;margin:0 auto;padding:24px;color:#222">

  <!-- Cabeçalho -->
  <div style="background:#FF6B35;padding:24px;border-radius:8px;margin-bottom:32px">
    <h1 style="color:#fff;margin:0;font-size:24px">📊 PR Score</h1>
    <p style="color:#fff;margin:4px 0 0;opacity:0.9">{DATA_CLIPPING}</p>
  </div>

  <!-- Placar resumido -->
  <div style="background:#f5f5f5;padding:16px;border-radius:8px;margin-bottom:32px;display:flex;gap:16px;flex-wrap:wrap">
    {''.join(f'<span style="font-size:15px">{EMOJIS_EMPRESA.get(e,"▪️")} <strong>{e}:</strong> {pontuacoes.get(e,0)} pts</span>' for e in EMPRESAS_PRINCIPAIS)}
  </div>

  <!-- Seções por empresa -->
  {secoes_html}

  <!-- Rodapé -->
  <div style="border-top:1px solid #eee;padding-top:16px;margin-top:32px;font-size:12px;color:#999;text-align:center">
    <p>PR Score · Loft · Time de PR<br>
    <a href="https://docs.google.com/spreadsheets/d/{DASHBOARD_SHEET_ID}" style="color:#FF6B35">Ver histórico completo no Dashboard</a></p>
  </div>

</body>
</html>"""

# Salvar HTML para download
data_arquivo = DATA_CLIPPING.replace("/", "-")
nome_arquivo = f"email_pr_score_{data_arquivo}.html"
with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write(html_email)

print("=" * 60)
print(f"📧 HTML DO E-MAIL salvo como: {nome_arquivo}")
print("=" * 60)

# Download automático
files.download(nome_arquivo)

print("\n✅ PROCESSO CONCLUÍDO!")
print(f"   Próximos passos:")
print(f"   1. Revisar matérias com 🔴 no Dashboard ({revisoes_pendentes} pendentes)")
print(f"   2. Enviar mensagem WhatsApp acima e aguardar validação")
print(f"   3. Após joinha do gestor: colar HTML no RD Station e enviar e-mail")
print(f"   4. Preencher coluna Retranca no Dashboard")